# TP MLOps II — Feature engineering y comparación de modelos

Continuación del baseline (`03_baseline.ipynb`). Objetivo inicial: enriquecer las features (variables cíclicas + lags de la demanda) y probar modelos no lineales para intentar superar al benchmark del TSO (`total load forecast`).

**Spoiler de lo que se descubre en el camino:** el enfoque inicial resulta estar "haciendo trampa" sin darse cuenta — `total load forecast` es un atajo casi perfecto, y el notebook termina rediseñando el problema para que sea un ejercicio de forecasting genuino.

## 1. Carga del dataset limpio

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

data_path = Path('../data')
df = pd.read_parquet(data_path / 'processed' / 'energy_weather_clean.parquet')
df.shape

(35064, 35)

## 2. Nuevas features

**Variables cíclicas (seno/coseno):** un modelo lineal trata `hour`, `dow` y `month` como números comunes, así que ve la hora 23 y la hora 0 como muy lejanas, cuando en realidad son consecutivas. Transformarlas a seno/coseno las "enrosca" en un círculo, resolviendo ese salto artificial en los bordes (23→0, diciembre→enero, domingo→lunes).

**Lags de la demanda (t-24h, t-168h):** la demanda de ayer a la misma hora, o de la semana pasada al mismo día/hora, es un predictor muy fuerte porque la gente repite rutinas (horario laboral, patrón de fin de semana). Son features legítimas sin leakage: al momento de predecir una hora futura, el pasado ya ocurrió y está disponible.

In [2]:
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
df['dow_sin'] = np.sin(2 * np.pi * df['dow'] / 7)
df['dow_cos'] = np.cos(2 * np.pi * df['dow'] / 7)

df['load_lag_24h'] = df['total load actual'].shift(24)
df['load_lag_168h'] = df['total load actual'].shift(168)

df[['total load actual', 'load_lag_24h', 'load_lag_168h']].isnull().sum()


total load actual      0
load_lag_24h          24
load_lag_168h        168
dtype: int64

## 3. Split temporal (ajustado)

Los lags generan nulos al principio de la serie (24 filas sin "ayer", 168 sin "semana pasada") porque no hay historia previa. Se dropean esas filas — caen todas dentro de 2015 (inicio de `train`), así que no afectan a `test` (2018).

In [3]:
df_model = df.dropna(subset=['load_lag_24h', 'load_lag_168h']) # Drop rows with NaN values in lag features

train = df_model[df_model.index.year < 2018]
test = df_model[df_model.index.year >= 2018]

train.shape, test.shape

((26137, 43), (8759, 43))

## 4. Primer intento: regresión lineal con las features nuevas

Se suman las cíclicas y los lags al set de features del baseline (calendario, clima, `total load forecast`), manteniendo el mismo modelo lineal para aislar el efecto de las features nuevas.

In [4]:
feature_cols = ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'is_weekend',
                 'temp_Madrid', 'temp_Barcelona', 'temp_Valencia', 'temp_Seville', 'temp_Bilbao',
                 'total load forecast',
                 'load_lag_24h', 'load_lag_168h']
target_col = 'total load actual'

X_train, y_train = train[feature_cols], train[target_col]
X_test, y_test = test[feature_cols], test[target_col]

X_train.shape, X_test.shape

((26137, 15), (8759, 15))

In [5]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, root_mean_squared_error

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

mae_lr = mean_absolute_error(y_test, y_pred)
mape_lr = mean_absolute_percentage_error(y_test, y_pred)
rmse_lr = root_mean_squared_error(y_test, y_pred)

print(f"MAE: {mae_lr:.1f} MW")
print(f"MAPE: {mape_lr:.2%}")
print(f"RMSE: {rmse_lr:.1f} MW")

MAE: 272.5 MW
MAPE: 0.93%
RMSE: 391.4 MW


**Resultado:** prácticamente sin cambio respecto al baseline (MAE 272.5 vs 271.8 MW). Las features nuevas no aportan nada apreciable.

## 5. Segundo intento: Random Forest

Como un modelo lineal no captura interacciones ni no linealidades (por ejemplo, cómo el efecto de la temperatura podría cambiar según la hora), se prueba un Random Forest con el mismo set de features, esperando que mejore.

In [6]:
from sklearn.ensemble import RandomForestRegressor

model_rf = RandomForestRegressor(n_estimators=200, max_depth=15, n_jobs=-1, random_state=42)
model_rf.fit(X_train, y_train)
y_pred_rf = model_rf.predict(X_test)

mae_rf = mean_absolute_error(y_test, y_pred_rf)
mape_rf = mean_absolute_percentage_error(y_test, y_pred_rf)
rmse_rf = root_mean_squared_error(y_test, y_pred_rf)

print(f"MAE: {mae_rf:.1f} MW")
print(f"MAPE: {mape_rf:.2%}")
print(f"RMSE: {rmse_rf:.1f} MW")

MAE: 282.9 MW
MAPE: 0.97%
RMSE: 401.8 MW


**Resultado inesperado:** el Random Forest queda *peor* que la regresión lineal (MAE 282.9 vs 272.5 MW). Para entender por qué, se mira qué feature está pesando más en las decisiones del árbol.

In [7]:
importances = pd.Series(model_rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
importances


total load forecast    0.992400
load_lag_168h          0.000950
load_lag_24h           0.000920
temp_Bilbao            0.000879
temp_Seville           0.000833
temp_Madrid            0.000740
temp_Barcelona         0.000677
temp_Valencia          0.000612
hour_cos               0.000446
hour_sin               0.000399
month_sin              0.000303
dow_sin                0.000301
month_cos              0.000286
dow_cos                0.000214
is_weekend             0.000041
dtype: float64

## 6. El hallazgo: `total load forecast` es un atajo, no una feature más

`total load forecast` concentra el **99.24%** de la importancia — el modelo ignora casi por completo el resto (clima, lags, calendario). Tiene sentido: esa columna es el **forecast oficial que el operador de red español publicó el día anterior** para cada hora, calculado con modelos internos mucho más sofisticados (con acceso a más información que la que hay en este dataset). Es casi la respuesta del problema, no un dato crudo — usarla no es leakage temporal (se publica con anticipación), pero **vuelve trivial el problema**: el modelo aprende a copiar al TSO en vez de aprender a predecir demanda a partir de sus verdaderos drivers.

Esto explica todo lo anterior: por qué el lineal ya empataba al benchmark en `03_baseline`, por qué las features nuevas no sumaban nada (ruido comparado con esa señal casi perfecta), y por qué el Random Forest empeoraba (particiona el espacio en vez de escalar linealmente la única variable que importa).

**Decisión:** sacar `total load forecast` del set de features. El objetivo deja de ser "casi copiar al TSO" y pasa a ser un problema de forecasting genuino: predecir la demanda a partir de calendario, clima e historia reciente — sin apoyarse en la predicción de otro.

## 7. Rediseño: features sin el atajo

In [8]:
feature_cols_v2 = ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'is_weekend',
                    'temp_Madrid', 'temp_Barcelona', 'temp_Valencia', 'temp_Seville', 'temp_Bilbao',
                    'load_lag_24h', 'load_lag_168h']

X_train_v2, y_train = train[feature_cols_v2], train[target_col]
X_test_v2, y_test = test[feature_cols_v2], test[target_col]


## 8. Comparación real: lineal vs. Random Forest (sin `total load forecast`)

In [9]:
model_lr_v2 = LinearRegression()
model_lr_v2.fit(X_train_v2, y_train)
y_pred_lr_v2 = model_lr_v2.predict(X_test_v2)

model_rf_v2 = RandomForestRegressor(n_estimators=200, max_depth=15, n_jobs=-1, random_state=42)
model_rf_v2.fit(X_train_v2, y_train)
y_pred_rf_v2 = model_rf_v2.predict(X_test_v2)

for name, y_pred in [('Linear', y_pred_lr_v2), ('RandomForest', y_pred_rf_v2)]:
    mae = mean_absolute_error(y_test, y_pred)
    mape = mean_absolute_percentage_error(y_test, y_pred)
    rmse = root_mean_squared_error(y_test, y_pred)
    print(f"{name}: MAE {mae:.1f} MW | MAPE {mape:.2%} | RMSE {rmse:.1f} MW")


Linear: MAE 2098.4 MW | MAPE 7.27% | RMSE 2860.7 MW
RandomForest: MAE 1751.3 MW | MAPE 6.04% | RMSE 2566.6 MW


**Ahora sí hay una historia clara:**
- El error sube fuerte respecto a cuando estaba el atajo (MAPE ~0.93% → ~6-7%), lo cual es esperado y correcto: sin la predicción del TSO, el problema es genuinamente más difícil.
- **Random Forest gana con margen** a la regresión lineal (MAE 1751.3 vs 2098.4 MW, ~17% mejor). Sin el atajo lineal dominante, las no linealidades e interacciones (como la relación en U entre temperatura y demanda, vista en el EDA) sí importan, y el modelo de árboles las aprovecha mejor.

## 9. Tercer modelo: Gradient Boosting

Se prueba también boosting, para ver si mejora sobre el Random Forest.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

model_gb = GradientBoostingRegressor(n_estimators=300, max_depth=4, learning_rate=0.05, random_state=42)
model_gb.fit(X_train_v2, y_train)
y_pred_gb = model_gb.predict(X_test_v2)

mae_gb = mean_absolute_error(y_test, y_pred_gb)
mape_gb = mean_absolute_percentage_error(y_test, y_pred_gb)
rmse_gb = root_mean_squared_error(y_test, y_pred_gb)

print(f"MAE: {mae_gb:.1f} MW")
print(f"MAPE: {mape_gb:.2%}")
print(f"RMSE: {rmse_gb:.1f} MW")


MAE: 1793.9 MW
MAPE: 6.22%
RMSE: 2593.6 MW


## 10. Resultado final

| Modelo | Features | MAE | MAPE |
|---|---|---|---|
| Regresión lineal | con `total load forecast` (atajo) | 272.5 MW | 0.93% |
| Random Forest | con `total load forecast` (atajo) | 282.9 MW | 0.97% |
| Regresión lineal | **sin** atajo | 2098.4 MW | 7.27% |
| Gradient Boosting | **sin** atajo | 1793.9 MW | 6.22% |
| **Random Forest** | **sin atajo** | **1751.3 MW** | **6.04%** |

**Random Forest, sin usar el forecast oficial del TSO como input, es el mejor modelo del notebook** (MAPE 6.04%). Es un resultado creíble y consistente con lo típico en literatura de load forecasting sin acceso al forecast de otro operador.

**Aprendizaje del proceso (más allá del número):** cuando una sola feature explica casi toda la varianza, vale la pena preguntarse si es una feature legítima o un atajo que vacía de sentido al problema — `feature_importances_` fue la herramienta que lo dejó en evidencia acá.

**Próximos pasos posibles:** tunear hiperparámetros del Random Forest o Gradient Boosting, probar XGBoost/LightGBM, más feature engineering (ventanas móviles de temperatura, distancia a un punto de confort térmico); y en paralelo, empezar a pensar la infraestructura para servir el modelo (Sesión 1 del curso: API REST).